## 被積分関数と真値

In [1]:
import numpy as np

f1 = lambda x: 1.0 / (1.0 + 3.0 * x * x)
f2 = lambda x: np.sqrt(np.maximum(1.0 - x * x, 0.0))
f3 = lambda x: np.sin(np.pi * (x - np.pi) / 2) ** 2

problems = [("(1) 1/(1+3x^2)",   f1, 2 * np.pi / (3 * np.sqrt(3))),
            ("(2) sqrt(1-x^2)",  f2, np.pi / 2),
            ("(3) sin^2(pi(x-pi)/2)", f3, 1.0)]


## ガウス型積分公式（ルジャンドル多項式の零点をニュートン法で計算）

In [2]:
def legendre(n, x):
    # 漸化式 (k+1)P_{k+1} = (2k+1)x P_k - k P_{k-1} で P_n と P_n' を計算
    p_prev = np.ones_like(x)   # P_0
    p = x.copy()               # P_1
    for k in range(1, n):
        p_next = ((2*k + 1) * x * p - k * p_prev) / (k + 1)
        p_prev, p = p, p_next
    dp = n * (x * p - p_prev) / (x * x - 1)
    return p, dp

def gauss_legendre(n):
    # 零点をニュートン法で求め、重みは w_j = 2/((1-a^2) P_n'(a)^2)
    a = np.cos(np.pi * (np.arange(1, n+1) - 0.25) / (n + 0.5))
    for _ in range(100):
        p, dp = legendre(n, a)
        da = -p / dp
        a = a + da
        if np.max(np.abs(da)) < 1e-15:
            break
    p, dp = legendre(n, a)
    w = 2.0 / ((1 - a * a) * dp * dp)
    return a, w

def gauss(f, n):
    a, w = gauss_legendre(n)
    return np.sum(w * f(a))


## 台形則 / 二重指数変換 + 台形則

In [3]:
def trapezoid(f, n, a=-1.0, b=1.0):
    # n 点、n-1 等分
    x = np.linspace(a, b, n)
    h = (b - a) / (n - 1)
    y = f(x)
    return h * (0.5 * y[0] + y[1:-1].sum() + 0.5 * y[-1])

def de_trapezoid(f, n, L=3.0):
    # t = tanh(pi/2 sinh(th)) で変換し、[-L, L] を n-1 等分して台形則
    th = np.linspace(-L, L, n)
    h = 2 * L / (n - 1)
    s = np.sinh(th)
    t = np.tanh(np.pi / 2 * s)
    w = (np.pi / 2 * np.cosh(th)) / np.cosh(np.pi / 2 * s) ** 2
    y = f(t) * w
    return h * (0.5 * y[0] + y[1:-1].sum() + 0.5 * y[-1])


## 精度の比較（n = 3, 5, 7）

In [4]:
for name, f, exact in problems:
    print(f"{name}   exact = {exact:.15f}")
    print(f"  {'n':>2}  {'Gauss':>26}  {'Trapezoid':>26}  {'DE + Trapezoid':>26}")
    for n in [3, 5, 7]:
        vals = [gauss(f, n), trapezoid(f, n), de_trapezoid(f, n)]
        cells_ = "  ".join(f"{v:>12.8f} ({abs(v-exact):.1e})" for v in vals)
        print(f"  {n:>2}  {cells_}")
    print()


(1) 1/(1+3x^2)   exact = 1.209199576156145
   n                       Gauss                   Trapezoid              DE + Trapezoid
   3    1.28571429 (7.7e-02)    1.25000000 (4.1e-02)    4.71238898 (3.5e+00)
   5    1.21764706 (8.4e-03)    1.19642857 (1.3e-02)    2.37000328 (1.2e+00)
   7    1.21014177 (9.4e-04)    1.20238095 (6.8e-03)    1.69475373 (4.9e-01)

(2) sqrt(1-x^2)   exact = 1.570796326794897
   n                       Gauss                   Trapezoid              DE + Trapezoid
   3    1.59161726 (2.1e-02)    1.00000000 (5.7e-01)    4.71238898 (3.1e+00)
   5    1.57590633 (5.1e-03)    1.36602540 (2.0e-01)    2.36007167 (7.9e-01)
   7    1.57278196 (2.0e-03)    1.45877669 (1.1e-01)    1.71251983 (1.4e-01)

(3) sin^2(pi(x-pi)/2)   exact = 1.000000000000000
   n                       Gauss                   Trapezoid              DE + Trapezoid
   3    1.02026908 (2.0e-02)    1.00000000 (0.0e+00)    4.48309677 (3.5e+00)
   5    1.00002781 (2.8e-05)    1.00000000 (0.0e+00)   

## 考察

- **(1)** 被積分関数は $[-1,1]$ 上で滑らかだが $x=\pm i/\sqrt3$ に極を持つ。2n−1 次まで厳密なガウス型が最も速く収束。台形則は Euler–Maclaurin より $O(h^2)$。
- **(2)** 端点 $x=\pm1$ で微分が発散する（$\sqrt{1-x^2}$）。多項式近似が効かずガウス型・台形則とも収束が遅い。二重指数変換は端点を引き伸ばして特異性を潰す手法だが、$n\le7$ では標本点が粗すぎて効果が出ない。
- **(3)** $f$ は周期 2 の周期関数で $f^{(2k-1)}(1)=f^{(2k-1)}(-1)$。Euler–Maclaurin の補正項が全て消えるため台形則が驚異的に高精度。

点数を増やした場合（(2) で二重指数変換が効いてくる）:

In [5]:
name, f, exact = problems[1]
print(f"{name}   exact = {exact:.15f}")
print(f"  {'n':>3}  {'Gauss':>14}  {'Trapezoid':>14}  {'DE + Trapezoid':>14}")
for n in [9, 17, 33, 65]:
    e = [abs(gauss(f, n) - exact), abs(trapezoid(f, n) - exact), abs(de_trapezoid(f, n) - exact)]
    print(f"  {n:>3}  " + "  ".join(f"{v:>14.2e}" for v in e))


(2) sqrt(1-x^2)   exact = 1.570796326794897
    n           Gauss       Trapezoid  DE + Trapezoid
    9        9.71e-04        7.29e-02        1.63e-02
   17        1.54e-04        2.59e-02        4.98e-07
   33        2.19e-05        9.17e-03        8.88e-16
   65        2.93e-06        3.25e-03        2.22e-16
